# Group 3 — Multilingual Tweet Sentiment Analysis
## Complete Practical Notebook (Stages 1–10)

**Environment:** Google Colab (GPU recommended)  
**Languages:** Swahili, Amharic, English  
**Target classes:** negative, neutral, positive

This notebook retains the executed Stage 1 acquisition/validation work and adds the remaining pipeline: preprocessing, mBERT/XLM-R tokenization, XLM-R fine-tuning, an LSTM baseline, evaluation, attention inspection, ablations and cross-lingual testing.

**Dataset provenance:** the supplied Stage 1 outputs use AfriSenti for Amharic and Swahili and TweetEval (`sentiment`) for English. English is not an AfriSenti language in the benchmark used here; this source distinction is retained throughout the notebook and report.

**Execution and evidence note:** The notebook retains outputs for dataset exploration, the main XLM-RoBERTa evaluation and the BiLSTM baseline. The ablation and cross-lingual sections are included as executable experiments; run those cells to regenerate their metric tables and figures. The accompanying report summarizes the measured experiment outputs supplied with this notebook. For a fresh end-to-end run, execute the cells in order in Google Colab; a GPU is recommended.


## 1. Dataset sources and known metadata

| Dataset | Languages used | Source splits | Text field | Label field |
|---|---|---|---|---|
| AfriSenti | Amharic, Swahili | train, dev, test | `tweet` | `label` |
| TweetEval (`sentiment`) | English | train, validation, test | `text` | `label` |

AfriSenti's Hugging Face dataset card describes 14 African languages and three sentiment classes. Its dataset-loading script uses the Amharic config `amh`; the upstream repository documentation also contains the alias `ama`. This notebook tries both raw-data paths for Amharic and records the successful one. Swahili uses `swa`.

TweetEval's `sentiment` configuration is English-only and maps class IDs as `0=negative`, `1=neutral`, `2=positive`. AfriSenti's loader defines its class names in a different order (`positive`, `neutral`, `negative`). The canonical project order below is therefore set explicitly as `negative=0`, `neutral=1`, `positive=2`.

**Licensing note:** the AfriSenti Hugging Face metadata and upstream repository have displayed different license labels. TweetEval's overall repository also notes that subset licenses differ. Check the current dataset cards and course requirements before redistributing raw data. Cite the original dataset papers in the report.

In [ ]:

# 2. Install Colab-compatible NLP dependencies

%pip -q install --upgrade --upgrade-strategy only-if-needed \
    "huggingface-hub>=1.32,<2" \
    "datasets>=5.2,<6" \
    "transformers==5.23.0" \
    "tokenizers>=0.23.1,<0.24" \
    "accelerate>=1.1" \
    "emoji>=2.12" \
    "scikit-learn>=1.4" \
    "sentencepiece>=0.2" \
    "pandas==2.2.3" \
    "numpy>=2.0,<2.5" \
    "requests==2.32.4" \
    "matplotlib>=3.7" \
    "seaborn>=0.13" \
    "pyarrow>=14" \
    "gradio>=6.32,<7" \
    "pillow>=13,<14"


In [ ]:
# 3. Imports and reproducibility settings

import os
import re
import json
import hashlib
import platform
from io import StringIO
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import datasets

from datasets import load_dataset
from IPython.display import display, Markdown

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid")

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("datasets:", datasets.__version__)
print("NumPy:", np.__version__)

In [ ]:
# 4. Project configuration

AFRISENTI_REPO = "https://raw.githubusercontent.com/afrisenti-semeval/afrisent-semeval-2023/main/data"
TWEETEVAL_REPO = "cardiffnlp/tweet_eval"
TWEETEVAL_CONFIG = "sentiment"

# Try the loader's official config code first, then the alias found in some upstream docs.
AFRISENTI_LANGUAGES = {
    "amharic": ["amh", "ama"],
    "swahili": ["swa"],
}

# Single canonical label mapping for every dataset/model in this project.
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {value: key for key, value in LABEL2ID.items()}
VALID_LABELS = set(LABEL2ID)

# Keep generated CSVs, manifest, checkpoints and plots in persistent Google Drive storage.
# In non-Colab Python, use a local folder instead.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    OUTPUT_DIR = Path('/content/drive/MyDrive/Group3_Sentiment_Project/stage1_outputs')
except ModuleNotFoundError:
    OUTPUT_DIR = Path('./group3_stage1_outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)
print("Canonical labels:", LABEL2ID)
print("Canonical labels:", LABEL2ID)

## 2. Acquire AfriSenti (Amharic and Swahili)

The upstream AfriSenti files are tab-separated files named `train.tsv`, `dev.tsv`, and `test.tsv`. This code downloads them directly from the official shared-task repository. The loader validates the expected columns and retains the original label text.

In [ ]:
# 5. Download and inspect AfriSenti raw TSV files.
# GitHub raw occasionally returns transient 503 responses; retry with backoff
# and fall back to the GitHub Contents API before failing the split.

import base64
import time
from requests.exceptions import RequestException

AFRI_SPLIT_FILES = {
    "train": "train.tsv",
    "validation": "dev.tsv",
    "test": "test.tsv",
}

GITHUB_OWNER = "afrisenti-semeval"
GITHUB_REPO = "afrisent-semeval-2023"
GITHUB_BRANCH = "main"
TRANSIENT_HTTP = {429, 500, 502, 503, 504}


def _get_with_retry(url: str, headers=None, attempts: int = 5):
    """GET with exponential backoff for transient network/server errors."""
    last_error = None
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=headers, timeout=(20, 90))
            if response.status_code == 404:
                return response
            if response.status_code in TRANSIENT_HTTP:
                retry_after = response.headers.get("Retry-After")
                try:
                    delay = float(retry_after) if retry_after else min(20, 2 ** attempt)
                except (TypeError, ValueError):
                    delay = min(20, 2 ** attempt)
                last_error = RuntimeError(
                    f"HTTP {response.status_code} from {url}; retry {attempt + 1}/{attempts}"
                )
                if attempt < attempts - 1:
                    print(f"Temporary HTTP {response.status_code}; retrying in {delay:.0f}s...")
                    time.sleep(delay)
                    continue
                break
            response.raise_for_status()
            return response
        except RequestException as exc:
            last_error = exc
            if attempt < attempts - 1:
                delay = min(20, 2 ** attempt)
                print(f"Temporary request error ({type(exc).__name__}); retrying in {delay:.0f}s...")
                time.sleep(delay)
    raise RuntimeError(f"GET failed after {attempts} attempts: {url}. Last error: {last_error}")


def _validate_afrisenti_tsv(text: str, url: str) -> pd.DataFrame:
    frame = pd.read_csv(StringIO(text), sep="\t")
    frame.columns = [str(column).strip().lower() for column in frame.columns]
    if not {"tweet", "label"}.issubset(frame.columns):
        raise ValueError(f"Unexpected columns at {url}: {frame.columns.tolist()}")
    if frame.empty:
        raise ValueError(f"Downloaded an empty TSV from {url}")
    return frame


def download_afrisenti_split(
    language_name: str,
    candidate_codes: list[str],
    split_name: str,
) -> tuple[pd.DataFrame, str, str]:
    """Download one upstream AfriSenti split with retries and alternate GitHub routes."""
    filename = AFRI_SPLIT_FILES[split_name]
    errors = []

    api_headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "Group3-AfriSenti-Downloader/1.0",
    }
    raw_headers = {
        "Accept": "text/plain",
        "User-Agent": "Group3-AfriSenti-Downloader/1.0",
    }

    for lang_code in candidate_codes:
        relative_path = f"data/{lang_code}/{filename}"
        api_url = (
            f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
            f"/contents/{relative_path}?ref={GITHUB_BRANCH}"
        )
        raw_url = f"{AFRISENTI_REPO}/{lang_code}/{filename}"
        github_raw_url = (
            f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"
            f"/raw/refs/heads/{GITHUB_BRANCH}/{relative_path}"
        )

        # 1) GitHub Contents API (separate route from raw.githubusercontent.com).
        try:
            response = _get_with_retry(api_url, headers=api_headers, attempts=4)
            if response.status_code == 404:
                errors.append(f"{lang_code} GitHub API: HTTP 404")
            else:
                payload = response.json()
                if payload.get("encoding") == "base64" and payload.get("content"):
                    tsv_text = base64.b64decode(payload["content"]).decode("utf-8-sig")
                elif payload.get("download_url"):
                    raw_response = _get_with_retry(payload["download_url"], headers=raw_headers, attempts=4)
                    if raw_response.status_code == 404:
                        raise FileNotFoundError(payload["download_url"])
                    tsv_text = raw_response.text
                else:
                    raise ValueError(f"GitHub API returned no file content for {relative_path}")
                frame = _validate_afrisenti_tsv(tsv_text, api_url)
                return frame, lang_code, api_url
        except Exception as exc:
            errors.append(f"{lang_code} GitHub API: {type(exc).__name__}: {exc}")

        # 2) Browser-style GitHub raw route.
        # 3) Original raw.githubusercontent.com URL.
        for fallback_url in [github_raw_url, raw_url]:
            try:
                response = _get_with_retry(fallback_url, headers=raw_headers, attempts=4)
                if response.status_code == 404:
                    errors.append(f"{lang_code} {fallback_url}: HTTP 404")
                    continue
                frame = _validate_afrisenti_tsv(response.text, fallback_url)
                return frame, lang_code, fallback_url
            except Exception as exc:
                errors.append(f"{lang_code} {fallback_url}: {type(exc).__name__}: {exc}")

    raise RuntimeError(
        f"Could not download AfriSenti {language_name}/{split_name}. "
        f"Tried language codes {candidate_codes}. " + " | ".join(errors)
    )


afri_raw = {}
afri_source_urls = {}
afri_used_codes = {}

for language_name, candidate_codes in AFRISENTI_LANGUAGES.items():
    afri_raw[language_name] = {}
    afri_source_urls[language_name] = {}
    successful_codes = set()

    for split_name in ["train", "validation", "test"]:
        raw_df, successful_code, source_url = download_afrisenti_split(
            language_name, candidate_codes, split_name
        )
        afri_raw[language_name][split_name] = raw_df
        afri_source_urls[language_name][split_name] = source_url
        successful_codes.add(successful_code)
        print(
            f"AfriSenti | {language_name:8s} | {split_name:10s} "
            f"| rows={len(raw_df):,} | code={successful_code}"
        )

    if len(successful_codes) != 1:
        raise RuntimeError(
            f"AfriSenti code changed between splits for {language_name}: {successful_codes}"
        )
    afri_used_codes[language_name] = successful_codes.pop()

for language_name in afri_raw:
    print(f"\\n{language_name.upper()} — train sample")
    display(afri_raw[language_name]["train"].head(3))


## 3. Acquire English sentiment data from TweetEval

Only the `sentiment` configuration is loaded. Do not accidentally load TweetEval's `emotion`, `hate`, `offensive`, `irony`, `emoji`, or stance tasks: those are different classification tasks and do not share the required three sentiment labels.

In [ ]:
# 6. Load TweetEval sentiment splits from Hugging Face

tweet_eval = load_dataset(TWEETEVAL_REPO, TWEETEVAL_CONFIG)

required_tweet_splits = {"train", "validation", "test"}
missing_tweet_splits = required_tweet_splits - set(tweet_eval.keys())
if missing_tweet_splits:
    raise ValueError(f"TweetEval is missing expected splits: {sorted(missing_tweet_splits)}")

print(tweet_eval)
print("TweetEval feature schema:", tweet_eval["train"].features)
display(pd.DataFrame(tweet_eval["train"][:5]))

## 4. Normalize schemas and labels safely

All data will be converted into the following schema:

| Column | Meaning |
|---|---|
| `text` | Original tweet text (unchanged) |
| `label` | Canonical class name: negative / neutral / positive |
| `label_id` | Canonical integer: 0 / 1 / 2 |
| `language` | `amharic`, `swahili`, or `english` |
| `dataset` | `AfriSenti` or `TweetEval` |
| `source_split` | `train`, `validation`, or `test` |
| `raw_label` | Original value from the source dataset |

The normalization function uses the Hugging Face `ClassLabel` names when they are available. This avoids a silent class inversion when datasets encode their class IDs differently.

In [ ]:
# 7. Label conversion helpers and canonical dataframe builder

LABEL_ALIASES = {
    "negative": "negative", "neg": "negative",
    "neutral": "neutral", "neu": "neutral",
    "positive": "positive", "pos": "positive",
}


def get_class_names(dataset_split) -> list[str] | None:
    """Return ClassLabel names when the split exposes a Hugging Face ClassLabel feature."""
    feature = dataset_split.features.get("label") if hasattr(dataset_split, "features") else None
    names = getattr(feature, "names", None)
    return list(names) if names is not None else None


def normalize_label(raw_label, class_names: list[str] | None = None) -> str:
    """Normalize a string or encoded class ID to one of the three canonical labels."""
    if pd.isna(raw_label):
        raise ValueError("Missing sentiment label encountered")

    value = raw_label
    if isinstance(value, (int, np.integer)) or (isinstance(value, float) and value.is_integer()):
        value = int(value)
        if class_names is None:
            raise ValueError(
                f"Integer label {value} has no ClassLabel names. Refusing to guess its meaning."
            )
        if value < 0 or value >= len(class_names):
            raise ValueError(f"Label ID {value} is outside the available class names: {class_names}")
        value = class_names[value]

    normalized = str(value).strip().lower()
    normalized = LABEL_ALIASES.get(normalized, normalized)
    if normalized not in VALID_LABELS:
        raise ValueError(f"Unexpected sentiment label {raw_label!r} (resolved to {normalized!r})")
    return normalized


def make_canonical_frame(
    frame: pd.DataFrame,
    *,
    text_column: str,
    label_column: str,
    language: str,
    dataset_name: str,
    split_name: str,
    class_names: list[str] | None = None,
) -> pd.DataFrame:
    """Create the project schema while preserving the original text and raw label."""
    missing_columns = {text_column, label_column} - set(frame.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns {sorted(missing_columns)} in {dataset_name}")

    output = pd.DataFrame({
        "text": frame[text_column].astype("string"),
        "raw_label": frame[label_column].copy(),
    })
    output["label"] = [normalize_label(x, class_names) for x in output["raw_label"]]
    output["label_id"] = output["label"].map(LABEL2ID).astype("int64")
    output["language"] = language
    output["dataset"] = dataset_name
    output["source_split"] = split_name
    return output[["text", "label", "label_id", "language", "dataset", "source_split", "raw_label"]]


canonical_parts = []

# AfriSenti TSV labels are strings in the official raw files.
for language_name, split_frames in afri_raw.items():
    for split_name, frame in split_frames.items():
        canonical_parts.append(make_canonical_frame(
            frame,
            text_column="tweet",
            label_column="label",
            language=language_name,
            dataset_name="AfriSenti",
            split_name=split_name,
            class_names=None,
        ))

# TweetEval labels are integer ClassLabel IDs; resolve them through the feature metadata.
for split_name in ["train", "validation", "test"]:
    split_data = tweet_eval[split_name]
    class_names = get_class_names(split_data)
    if class_names is None:
        raise ValueError("TweetEval label feature did not expose ClassLabel names; inspect schema before continuing.")
    print(f"TweetEval class ID mapping from dataset metadata: {dict(enumerate(class_names))}")
    tweet_frame = split_data.to_pandas()
    canonical_parts.append(make_canonical_frame(
        tweet_frame,
        text_column="text",
        label_column="label",
        language="english",
        dataset_name="TweetEval",
        split_name=split_name,
        class_names=class_names,
    ))

all_data = pd.concat(canonical_parts, ignore_index=True)

print("Canonical data shape:", all_data.shape)
display(all_data.head())
print("Canonical class mapping:", LABEL2ID)

## 5. Validate completeness, text quality and label coverage

This section checks that all required languages and all three sentiment labels are present in each expected split. It reports null/blank text values and unexpected labels. It does not silently delete suspicious records.

In [ ]:
# 8. Core validation checks

EXPECTED_DATASETS = {
    ("AfriSenti", "amharic"),
    ("AfriSenti", "swahili"),
    ("TweetEval", "english"),
}
EXPECTED_SPLITS = {"train", "validation", "test"}

actual_dataset_languages = set(zip(all_data["dataset"], all_data["language"]))
assert EXPECTED_DATASETS.issubset(actual_dataset_languages), (
    f"Missing dataset-language combinations: {EXPECTED_DATASETS - actual_dataset_languages}"
)

actual_splits = set(all_data["source_split"].unique())
assert EXPECTED_SPLITS.issubset(actual_splits), f"Missing expected splits: {EXPECTED_SPLITS - actual_splits}"

# Missing and blank text
missing_text_count = int(all_data["text"].isna().sum())
blank_text_count = int(all_data["text"].fillna("").str.strip().eq("").sum())
missing_label_count = int(all_data["label"].isna().sum())
unknown_labels = sorted(set(all_data["label"].dropna().unique()) - VALID_LABELS)

quality_summary = pd.DataFrame({
    "check": ["rows", "missing text", "blank text", "missing canonical label", "unknown labels"],
    "result": [len(all_data), missing_text_count, blank_text_count, missing_label_count, unknown_labels],
})
display(quality_summary)

assert missing_text_count == 0, "Missing text values found; inspect before proceeding."
assert blank_text_count == 0, "Blank tweets found; inspect before proceeding."
assert missing_label_count == 0, "Missing labels found; inspect before proceeding."
assert not unknown_labels, f"Unexpected labels found: {unknown_labels}"

# Every expected split should contain all three classes. If this fails, inspect the source split.
coverage_rows = []
for (dataset_name, language, split_name), part in all_data.groupby(["dataset", "language", "source_split"]):
    present = set(part["label"].unique())
    coverage_rows.append({
        "dataset": dataset_name,
        "language": language,
        "split": split_name,
        "rows": len(part),
        "labels_present": ", ".join(sorted(present)),
        "missing_classes": ", ".join(sorted(VALID_LABELS - present)),
    })

class_coverage = pd.DataFrame(coverage_rows).sort_values(["dataset", "language", "split"])
display(class_coverage)

## 6. Split sizes and label distribution

The tables below describe the data actually downloaded in this run. Published counts are useful as a sanity check, but do not hard-code them as guaranteed truths: repository revisions, aliases or dataset packaging may change. Record the counts produced by this notebook in the report.

In [ ]:
# 9. Dataset size summary

split_summary = (
    all_data.groupby(["dataset", "language", "source_split"], observed=True)
    .size()
    .rename("number_of_tweets")
    .reset_index()
    .sort_values(["dataset", "language", "source_split"])
)
display(split_summary)

# Per-split sentiment counts and proportions
label_counts = (
    all_data.groupby(["dataset", "language", "source_split", "label"], observed=True)
    .size()
    .rename("count")
    .reset_index()
)
label_counts["proportion"] = label_counts["count"] / label_counts.groupby(
    ["dataset", "language", "source_split"]
)["count"].transform("sum")
display(label_counts)

In [ ]:
# 10. Visualize sentiment distribution by language and split

plot_data = label_counts.copy()
plot_data["group"] = plot_data["language"] + " | " + plot_data["source_split"]

plt.figure(figsize=(13, 6))
sns.barplot(data=plot_data, x="group", y="count", hue="label", hue_order=["negative", "neutral", "positive"])
plt.title("Sentiment class counts by language and split")
plt.xlabel("Language | split")
plt.ylabel("Number of tweets")
plt.xticks(rotation=35, ha="right")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

plt.figure(figsize=(13, 6))
sns.barplot(data=plot_data, x="group", y="proportion", hue="label", hue_order=["negative", "neutral", "positive"])
plt.title("Sentiment class proportions by language and split")
plt.xlabel("Language | split")
plt.ylabel("Proportion")
plt.ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

## 7. Exact duplicates and possible split leakage

Exact duplicate text can occur naturally, especially in social-media corpora. We therefore **report** duplicates rather than automatically deleting them. A text occurring across train and test deserves manual attention because it can inflate evaluation results.

In [ ]:
# 11. Duplicate analysis within each dataset-language collection

def duplicate_report(group: pd.DataFrame) -> pd.Series:
    duplicate_mask = group["text"].duplicated(keep=False)
    split_sets = group.loc[duplicate_mask].groupby("text")["source_split"].agg(lambda values: sorted(set(values)))
    cross_split = int(split_sets.map(len).gt(1).sum())
    return pd.Series({
        "rows": len(group),
        "duplicate_rows_including_repeats": int(duplicate_mask.sum()),
        "unique_texts_repeated": int(group.loc[duplicate_mask, "text"].nunique()),
        "duplicate_texts_across_multiple_splits": cross_split,
    })


duplicate_rows = []
for (dataset_name, language), group in all_data.groupby(["dataset", "language"]):
    result = duplicate_report(group).to_dict()
    result["dataset"] = dataset_name
    result["language"] = language
    duplicate_rows.append(result)

duplicate_summary = pd.DataFrame(duplicate_rows)[
    ["dataset", "language", "rows", "duplicate_rows_including_repeats",
     "unique_texts_repeated", "duplicate_texts_across_multiple_splits"]
]
display(duplicate_summary)

# Show a small number of exact duplicate examples that span different splits.
for (dataset_name, language), group in all_data.groupby(["dataset", "language"]):
    split_sets = group.groupby("text")["source_split"].agg(lambda values: sorted(set(values)))
    overlapping_texts = split_sets[split_sets.map(len) > 1].index
    if len(overlapping_texts):
        print(f"\nCross-split duplicate examples — {dataset_name}/{language}")
        display(group[group["text"].isin(overlapping_texts)].sort_values(["text", "source_split"]).head(10))

## 8. Text-length and character-level sanity checks

This stage records raw text properties without applying cleaning. Length is measured in Unicode characters, not tokenizer tokens. Token-length analysis will be repeated later using the actual multilingual tokenizer.

In [ ]:
# 12. Raw text length summary

all_data["character_length"] = all_data["text"].astype(str).str.len()
length_summary = (
    all_data.groupby(["dataset", "language", "source_split"])["character_length"]
    .agg(count="count", mean="mean", median="median", min="min", max="max", std="std")
    .reset_index()
)
display(length_summary.round(2))

plt.figure(figsize=(12, 6))
sns.boxplot(data=all_data, x="language", y="character_length", hue="source_split", showfliers=False)
plt.title("Raw tweet character length by language and split")
plt.xlabel("Language")
plt.ylabel("Characters per tweet")
plt.tight_layout()
plt.show()

## 9. Inspect examples without changing them

These examples are for qualitative inspection only. They help identify script differences, emoji presence, URLs, mentions, hashtags and possible code-switching. Avoid copying identifiable or sensitive tweet content into the final report; aggregate observations are usually sufficient.

In [ ]:
# 13. Random examples by language and sentiment

for language in ["amharic", "swahili", "english"]:
    print(f"\n===== {language.upper()} =====")
    subset = all_data[(all_data["language"] == language) & (all_data["source_split"] == "train")]
    if subset.empty:
        print("No examples available")
        continue
    sample_n = min(3, len(subset))
    sample_parts = []
    for _, label_group in subset.groupby("label"):
        sample_parts.append(label_group.sample(n=1, random_state=RANDOM_SEED))
    sample = pd.concat(sample_parts, ignore_index=True).head(sample_n) if sample_parts else subset.head(0)
    display(sample[["text", "label", "dataset", "language"]])

## 10. Save the validated Stage 1 data and manifest

The notebook saves:
- one canonical CSV for each dataset/language/split;
- one combined CSV per split (never combine splits together);
- a JSON manifest containing source URLs, dataset identifiers, package versions, retrieval time, counts and canonical label mapping.

In Google Colab, files are saved under `/content/drive/MyDrive/Group3_Sentiment_Project/stage1_outputs`. Outside Colab, a local `group3_stage1_outputs` folder is used. Re-run the acquisition and save cells after changing the output path so old cached outputs are refreshed.

In [ ]:
# 14. Save separate source/language/split datasets

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for (dataset_name, language, split_name), frame in all_data.groupby(
    ["dataset", "language", "source_split"], sort=True
):
    safe_dataset = re.sub(r"[^a-zA-Z0-9_-]+", "_", dataset_name.lower())
    output_path = OUTPUT_DIR / f"{safe_dataset}_{language}_{split_name}.csv"
    frame.drop(columns=["character_length"], errors="ignore").to_csv(output_path, index=False, encoding="utf-8")

# Combined files are split-specific and retain dataset/language provenance.
for split_name in ["train", "validation", "test"]:
    frame = all_data[all_data["source_split"] == split_name].copy()
    frame.drop(columns=["character_length"], errors="ignore").to_csv(
        OUTPUT_DIR / f"combined_{split_name}.csv", index=False, encoding="utf-8"
    )

print("Saved CSV files:")
for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"{path.name:45s} {path.stat().st_size:>10,} bytes")

In [ ]:
# 15. Build a retrieval manifest for reproducibility

manifest = {
    "project": "Group 3 Multilingual Tweet Sentiment Analysis",
    "stage": "Stage 1 — Dataset Acquisition and Validation",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "package_versions": {
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "datasets": datasets.__version__,
    },
    "sources": {
        "AfriSenti": {
            "repository": "https://github.com/afrisenti-semeval/afrisent-semeval-2023",
            "raw_base_url": AFRISENTI_REPO,
            "language_codes_used": afri_used_codes,
            "source_urls": afri_source_urls,
        },
        "TweetEval": {
            "huggingface_dataset": TWEETEVAL_REPO,
            "configuration": TWEETEVAL_CONFIG,
            "revision": "main",
        },
    },
    "language_sources": {
        "amharic": "AfriSenti",
        "swahili": "AfriSenti",
        "english": "TweetEval",
    },
    "canonical_label_mapping": LABEL2ID,
    "row_counts": split_summary.to_dict(orient="records"),
    "notes": [
        "English comes from TweetEval, not AfriSenti.",
        "Labels were normalized by their names, not by raw integer IDs.",
        "Raw tweet text was not cleaned or normalized in Stage 1.",
        "The Hugging Face dataset and upstream repository metadata should be rechecked before redistribution.",
    ],
}

manifest_path = OUTPUT_DIR / "stage1_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2, default=str)

print("Manifest saved to:", manifest_path)

## 11. Final Stage 1 acceptance checks

Stage 1 is complete only when all checks below pass:

- [ ] Amharic AfriSenti train, validation and test splits downloaded.
- [ ] Swahili AfriSenti train, validation and test splits downloaded.
- [ ] English TweetEval sentiment train, validation and test splits downloaded.
- [ ] All three canonical labels are present and correctly mapped.
- [ ] No missing or blank text rows remain unnoticed.
- [ ] Duplicate and cross-split duplicate counts have been reviewed.
- [ ] Data counts and sentiment distributions have been recorded.
- [ ] CSV outputs and `stage1_manifest.json` have been saved.

**Continue to Stage 2 only after the checks above pass.**

The cached outputs in the originally uploaded notebook were generated before the Drive-persistence patch and may show the earlier `/content/group3_stage1_outputs` path. Run the notebook from the beginning in Colab to refresh all Stage 1 outputs and save the files to Drive.

### Stage 1 deliverables
1. `combined_train.csv`, `combined_validation.csv`, `combined_test.csv`
2. Individual dataset/language/split CSV files
3. `stage1_manifest.json`
4. Tables and plots visible in the executed notebook

### References
- AfriSenti dataset card: https://huggingface.co/datasets/HausaNLP/AfriSenti-Twitter
- AfriSenti shared-task repository: https://github.com/afrisenti-semeval/afrisent-semeval-2023
- AfriSenti paper: Muhammad et al. (2023), *AfriSenti: A Twitter Sentiment Analysis Benchmark for African Languages*. https://aclanthology.org/2023.emnlp-main.862/
- TweetEval dataset card: https://huggingface.co/datasets/cardiffnlp/tweet_eval
- TweetEval paper: Barbieri et al. (2020), *TweetEval: Unified Benchmark and Comparative Evaluation for Tweet Classification*. https://aclanthology.org/2020.findings-emnlp.148/

---
# Stage 2 — Text preprocessing and multilingual tokenization

This stage preserves the original `text` column and creates a separate `model_text` field. URLs and user mentions are replaced with explicit placeholders; emoji are converted to readable text labels; Unicode is normalized; and slang substitutions are controlled by a small, editable language-specific dictionary. Do not add Swahili or Amharic slang mappings unless they have been checked by a fluent speaker. The validation and test sets receive exactly the same deterministic transformations as the training set; they are not used to invent normalization rules.


In [ ]:
# 2.1 Verify the modelling dependencies installed in Section 2.
# Do not reinstall Transformers here: the notebook consistently uses Transformers 5.x.
from importlib.metadata import version, PackageNotFoundError

for package_name in ["transformers", "huggingface-hub", "tokenizers", "datasets", "accelerate"]:
    try:
        print(f"{package_name}: {version(package_name)}")
    except PackageNotFoundError:
        print(f"{package_name}: NOT INSTALLED — rerun Section 2 installation")


In [ ]:
# Stage 2 imports and output folders
import gc
import html
import math
import random
import unicodedata
from collections import Counter

import emoji
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    roc_curve, auc,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FIGURES_DIR = OUTPUT_DIR / 'figures'
RESULTS_DIR = OUTPUT_DIR / 'model_results'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

required_columns = {'text', 'label', 'label_id', 'language', 'source_split', 'dataset'}
missing_columns = required_columns - set(all_data.columns)
if missing_columns:
    raise ValueError(f'Stage 1 dataframe is missing required columns: {sorted(missing_columns)}')

print('Device:', DEVICE)
print('Output folder:', OUTPUT_DIR)
print('Figure folder:', FIGURES_DIR)
print('Rows available:', f'{len(all_data):,}')

## 2.1 EDA recap and split-safe training samples

The Stage 1 notebook already reports language/split sizes, sentiment counts, raw character lengths, missing values and duplicates. The cells below add a compact language-distribution plot and create language-balanced train/validation samples for model selection. The held-out test data are not downsampled for the final per-language evaluation.


In [ ]:
# Add explicit language-distribution and label-imbalance visuals from the loaded Stage 1 data.
lang_split = (all_data.groupby(['language', 'source_split']).size()
              .rename('tweets').reset_index())
plt.figure(figsize=(10, 5), dpi=160)
sns.barplot(data=lang_split, x='language', y='tweets', hue='source_split')
plt.title('Tweet counts by language and split')
plt.xlabel('Language')
plt.ylabel('Number of tweets')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'language_distribution_by_split.png', dpi=300, bbox_inches='tight')
plt.show()

train_eda = all_data[all_data['source_split'] == 'train'].copy()

# Calculate sentiment proportions by language
# Compatible with newer Pandas versions

label_counts = (
    train_eda.groupby(['language', 'label']).size()
)

language_totals = (
    label_counts.groupby(level='language').transform('sum')
)

label_mix = (
    (label_counts / language_totals)
    .rename('proportion')
    .reset_index()
)

print("Sentiment proportions by language:")
display(label_mix)

plt.figure(figsize=(9, 5), dpi=160)
sns.barplot(data=label_mix, x='language', y='proportion', hue='label',
            hue_order=['negative', 'neutral', 'positive'])
plt.title('Training sentiment-label proportions by language')
plt.xlabel('Language')
plt.ylabel('Within-language proportion')
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_label_proportions.png', dpi=300, bbox_inches='tight')
plt.show()

print('Training rows by language:')
display(train_eda.groupby('language').size().rename('train_rows').to_frame())
print('Training sentiment counts:')
display(pd.crosstab(train_eda['language'], train_eda['label']))

## 2.2 Unicode, URL, mention, emoji and slang normalization

The starter slang map contains only common English chat abbreviations. The Swahili and Amharic maps are intentionally empty until native-speaker-reviewed mappings are supplied. The code records how many replacements were made so the preprocessing report can describe what really happened.


In [ ]:
# Conservative, editable slang normalization dictionaries.
SLANG_MAP = {
    'english': {
        'u': 'you', 'ur': 'your', 'pls': 'please', 'plz': 'please',
        'idk': 'i do not know', 'imo': 'in my opinion', 'btw': 'by the way',
    },
    'swahili': {},   # Add only Swahili variants verified by a fluent speaker.
    'amharic': {},   # Add only Amharic variants verified by a fluent speaker.
}

URL_PATTERN = re.compile(r'(https?://\S+|www\.\S+)', flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r'(?<!\w)@\w+', flags=re.UNICODE)


def normalize_tweet(text, language):
    """Return normalized text and replacement counts; original text is untouched."""
    text = '' if pd.isna(text) else str(text)
    text = html.unescape(unicodedata.normalize('NFC', text))
    counts = {'urls': 0, 'mentions': 0, 'emoji_sequences': 0, 'slang_terms': 0}

    text, counts['urls'] = URL_PATTERN.subn(' URLTOKEN ', text)
    text, counts['mentions'] = MENTION_PATTERN.subn(' USERTOKEN ', text)

    # Convert emoji to readable names instead of deleting sentiment-bearing symbols.
    try:
        counts['emoji_sequences'] = sum(1 for _ in emoji.analyze(text))
        text = emoji.demojize(text, language='en', delimiters=(' EMOJI_', '_ '))
    except Exception:
        # Keep original emoji if an installed emoji version does not support analyze().
        text = emoji.demojize(text, language='en')

    slang = SLANG_MAP.get(str(language).lower(), {})
    for short_form, expanded_form in slang.items():
        pattern = re.compile(r'(?<!\w)' + re.escape(short_form) + r'(?!\w)', re.IGNORECASE | re.UNICODE)
        text, changed = pattern.subn(expanded_form, text)
        counts['slang_terms'] += changed

    text = re.sub(r'\s+', ' ', text).strip()
    return text, counts


model_data = all_data.copy()
normalization_totals = Counter()
normalized_values = []
for row in model_data[['text', 'language']].itertuples(index=False):
    normalized, counts = normalize_tweet(row.text, row.language)
    normalized_values.append(normalized)
    normalization_totals.update(counts)
model_data['model_text'] = normalized_values

print('Normalization replacement totals (actual run):')
display(pd.DataFrame([normalization_totals]))
print('Example before/after pairs (first 5 rows only):')
display(model_data[['language', 'text', 'model_text']].head(5))
model_data.to_csv(OUTPUT_DIR / 'all_splits_preprocessed.csv', index=False, encoding='utf-8')

## 2.3 Compare mBERT and XLM-RoBERTa tokenization

Both multilingual tokenizers are instantiated and their token-length distributions are measured on a reproducible sample from each language. The main fine-tuned classifier below uses XLM-RoBERTa. The tokenizer comparison is descriptive; tokenization is not the same as model performance.


In [ ]:
MBERT_NAME = 'bert-base-multilingual-cased'
XLMR_NAME = 'xlm-roberta-base'
TOKENIZER_SAMPLE_PER_LANGUAGE = 250

mbert_tokenizer = AutoTokenizer.from_pretrained(MBERT_NAME)
xlmr_tokenizer = AutoTokenizer.from_pretrained(XLMR_NAME)


def tokenizer_length_summary(frame, tokenizer, tokenizer_name):
    rows = []
    for language in ['swahili', 'amharic', 'english']:
        subset = frame[frame['language'] == language]
        if subset.empty:
            continue
        sample = subset.sample(n=min(TOKENIZER_SAMPLE_PER_LANGUAGE, len(subset)),
                               random_state=RANDOM_SEED)
        lengths = [len(tokenizer.encode(t, add_special_tokens=True, truncation=False))
                   for t in sample['model_text'].tolist()]
        rows.append({
            'tokenizer': tokenizer_name,
            'language': language,
            'n_sampled': len(lengths),
            'p50_tokens': float(np.percentile(lengths, 50)),
            'p90_tokens': float(np.percentile(lengths, 90)),
            'p95_tokens': float(np.percentile(lengths, 95)),
            'max_tokens': int(np.max(lengths)),
        })
    return rows

train_preprocessed = model_data[model_data['source_split'] == 'train'].copy()
token_length_rows = []
token_length_rows.extend(tokenizer_length_summary(train_preprocessed, mbert_tokenizer, 'mBERT'))
token_length_rows.extend(tokenizer_length_summary(train_preprocessed, xlmr_tokenizer, 'XLM-RoBERTa'))
token_length_df = pd.DataFrame(token_length_rows)
display(token_length_df)
token_length_df.to_csv(RESULTS_DIR / 'tokenizer_length_summary.csv', index=False)

# Chosen truncation length; vary this value in the ablation section below.
MAX_LENGTH = 128
print('Main XLM-RoBERTa maximum sequence length:', MAX_LENGTH)

# Stage 3 — XLM-RoBERTa fine-tuning (3-class classification)

The training sample is balanced across the three languages to prevent the largest English source from dominating batches. Within-language label proportions are approximately preserved during down-sampling. The validation sample is also language-balanced for model selection. Final test metrics are reported separately by language and on the original, non-downsampled held-out test partitions.

Training uses up to five epochs, early stopping with patience two, class-weighted loss and gradient clipping (`max_grad_norm=1.0`). The train/validation/test partitions remain separate.


In [ ]:
# Build reproducible language-balanced training and validation samples.
def language_balanced_sample(frame, language_col='language', label_col='label_id', seed=RANDOM_SEED):
    counts = frame.groupby(language_col).size()
    if counts.empty:
        raise ValueError('No rows available for language balancing.')
    n_per_language = int(counts.min())
    parts = []
    for language, group in frame.groupby(language_col):
        if len(group) == n_per_language:
            sampled = group.copy()
        else:
            # Stratified down-sampling keeps each language's label mix as close as possible.
            try:
                sampled, _ = train_test_split(
                    group, train_size=n_per_language, stratify=group[label_col],
                    random_state=seed
                )
            except ValueError:
                sampled = group.sample(n=n_per_language, random_state=seed)
        parts.append(sampled)
    result = pd.concat(parts, ignore_index=True)
    return result.sample(frac=1, random_state=seed).reset_index(drop=True)

train_df = model_data[model_data['source_split'] == 'train'].copy()
validation_df = model_data[model_data['source_split'] == 'validation'].copy()
test_df = model_data[model_data['source_split'] == 'test'].copy()

balanced_train_df = language_balanced_sample(train_df)
balanced_validation_df = language_balanced_sample(validation_df)
print('Original training rows by language:')
display(train_df.groupby('language').size().rename('rows').to_frame())
print('Balanced training rows by language:')
display(balanced_train_df.groupby('language').size().rename('rows').to_frame())
print('Balanced validation rows by language:')
display(balanced_validation_df.groupby('language').size().rename('rows').to_frame())

In [ ]:
# XLM-RoBERTa dataset builder and metric functions
LABEL_NAMES = ['negative', 'neutral', 'positive']
NUM_LABELS = len(LABEL_NAMES)


def to_hf_dataset(frame, tokenizer, max_length=MAX_LENGTH):
    small = frame[['model_text', 'label_id']].rename(
        columns={'model_text': 'text', 'label_id': 'labels'}
    ).reset_index(drop=True)
    dataset = HFDataset.from_pandas(small, preserve_index=False)

    def tokenize_batch(batch):
        return tokenizer(batch['text'], truncation=True, max_length=max_length)

    return dataset.map(tokenize_batch, batched=True, remove_columns=['text'])


def metric_bundle(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    y_pred = probabilities.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average='macro', zero_division=0
    )
    try:
        auc_ovr = roc_auc_score(
            y_true, probabilities, labels=[0, 1, 2],
            multi_class='ovr', average='macro'
        )
    except ValueError:
        auc_ovr = np.nan  # ROC-AUC is undefined if a required class is absent.
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision_macro': float(precision),
        'recall_macro': float(recall),
        'f1_macro': float(f1),
        'roc_auc_ovr_macro': float(auc_ovr) if np.isfinite(auc_ovr) else None,
    }, y_pred


def trainer_compute_metrics(eval_prediction):
    logits = eval_prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.asarray(logits)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    metrics, _ = metric_bundle(eval_prediction.label_ids, probs)
    # Trainer's early-stopping metric uses the macro F1 key.
    return {k: v for k, v in metrics.items() if v is not None}

In [ ]:
# XLM-RoBERTa Trainer factory. Call once for the main model and again for ablations/cross-lingual training.

def train_xlmr(
    train_frame,
    validation_frame,
    run_name='xlmr_main',
    model_name=XLMR_NAME,
    max_length=MAX_LENGTH,
    batch_size=8,
    learning_rate=2e-5,
    epochs=5,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_dataset = to_hf_dataset(train_frame, tokenizer, max_length=max_length)
    validation_dataset = to_hf_dataset(validation_frame, tokenizer, max_length=max_length)

    label_counts = np.bincount(train_frame['label_id'].astype(int), minlength=NUM_LABELS)
    class_weights = len(train_frame) / (NUM_LABELS * np.maximum(label_counts, 1))
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=NUM_LABELS,
        id2label={i: LABEL_NAMES[i] for i in range(NUM_LABELS)},
        label2id={LABEL_NAMES[i]: i for i in range(NUM_LABELS)},
        attn_implementation='eager',
    )

    class CostSensitiveTrainer(Trainer):
        def __init__(self, *args, class_weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = class_weights

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            # Do not mutate the batch dictionary: Trainer still needs its labels for metrics.
            labels = inputs['labels']
            model_inputs = {key: value for key, value in inputs.items() if key != 'labels'}
            outputs = model(**model_inputs)
            weights = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
            loss = F.cross_entropy(outputs.logits, labels, weight=weights)
            return (loss, outputs) if return_outputs else loss

    run_dir = RESULTS_DIR / run_name
    args = TrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=max(8, batch_size),
        gradient_accumulation_steps=2,
        learning_rate=learning_rate,
        weight_decay=0.01,
        eval_strategy='epoch',
        save_strategy='epoch',
        logging_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='f1_macro',
        greater_is_better=True,
        save_total_limit=2,
        max_grad_norm=1.0,
        fp16=torch.cuda.is_available(),
        report_to='none',
        seed=RANDOM_SEED,
        data_seed=RANDOM_SEED,
    )
    trainer = CostSensitiveTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=validation_dataset,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=trainer_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights,
    )
    trainer.train()
    trainer.save_model(str(run_dir / 'best_model'))
    tokenizer.save_pretrained(str(run_dir / 'best_model'))
    return trainer, tokenizer


MAIN_BATCH_SIZE = 8
MAIN_LEARNING_RATE = 2e-5
MAIN_EPOCHS = 5  # Early stopping may end training before epoch 5 (but not before 3 epochs).
xlmr_trainer, xlmr_tokenizer = train_xlmr(
    balanced_train_df, balanced_validation_df,
    run_name='xlmr_main', batch_size=MAIN_BATCH_SIZE,
    learning_rate=MAIN_LEARNING_RATE, max_length=MAX_LENGTH,
    epochs=MAIN_EPOCHS,
)

## 3.1 Test evaluation, confusion matrices, ROC curves and sample predictions

The following cells evaluate the selected XLM-RoBERTa checkpoint on the original held-out test subsets. Metrics are reported per language and overall. Macro-F1 gives each class equal weight; ROC-AUC is one-vs-rest macro AUC. Test partitions are not used for early stopping or hyperparameter selection.


In [ ]:
# Prediction helper for XLM-RoBERTa

def predict_xlmr(trainer, tokenizer, frame, max_length=MAX_LENGTH):
    dataset = to_hf_dataset(frame, tokenizer, max_length=max_length)
    prediction = trainer.predict(dataset)
    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    metrics, predictions = metric_bundle(frame['label_id'].to_numpy(), probabilities)
    return metrics, predictions, probabilities


xlmr_metrics = []
xlmr_prediction_frames = []
xlmr_outputs = {}
for language in ['swahili', 'amharic', 'english']:
    subset = test_df[test_df['language'] == language].reset_index(drop=True)
    metrics, predictions, probabilities = predict_xlmr(xlmr_trainer, xlmr_tokenizer, subset)
    row = {'model': 'XLM-RoBERTa', 'evaluation': f'{language}_test', 'n': len(subset), **metrics}
    xlmr_metrics.append(row)
    xlmr_outputs[language] = (subset, predictions, probabilities)
    part = subset[['text', 'model_text', 'language', 'label']].copy()
    part['predicted_label'] = [LABEL_NAMES[i] for i in predictions]
    part['confidence'] = probabilities.max(axis=1)
    xlmr_prediction_frames.append(part)

all_metrics, all_predictions, all_probabilities = predict_xlmr(
    xlmr_trainer, xlmr_tokenizer, test_df.reset_index(drop=True)
)
xlmr_metrics.append({'model':'XLM-RoBERTa', 'evaluation':'all_test', 'n':len(test_df), **all_metrics})
xlmr_metrics_df = pd.DataFrame(xlmr_metrics)
display(xlmr_metrics_df)
xlmr_metrics_df.to_csv(RESULTS_DIR / 'xlmr_test_metrics.csv', index=False)

xlmr_predictions_df = pd.concat(xlmr_prediction_frames, ignore_index=True)
xlmr_predictions_df.to_csv(RESULTS_DIR / 'xlmr_sample_predictions_all_test.csv', index=False)
print('Example predictions (10 rows):')
display(xlmr_predictions_df[['language', 'text', 'label', 'predicted_label', 'confidence']].head(10))

In [ ]:
# Confusion matrices and one-vs-rest ROC curves for each language plus the complete test set.

def save_evaluation_plots(y_true, y_pred, y_prob, title_prefix, file_prefix):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    plt.figure(figsize=(5.5, 4.5), dpi=160)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    plt.title(f'{title_prefix}: confusion matrix')
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{file_prefix}_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    y_bin = label_binarize(y_true, classes=[0, 1, 2])
    plt.figure(figsize=(6, 5), dpi=160)
    for class_id, class_name in enumerate(LABEL_NAMES):
        if len(np.unique(y_bin[:, class_id])) < 2:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, class_id], y_prob[:, class_id])
        class_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_name} (AUC={class_auc:.3f})')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Chance')
    plt.title(f'{title_prefix}: one-vs-rest ROC')
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'{file_prefix}_roc.png', dpi=300, bbox_inches='tight')
    plt.show()

for language, (subset, predictions, probabilities) in xlmr_outputs.items():
    save_evaluation_plots(subset['label_id'].to_numpy(), predictions, probabilities,
                          f'XLM-RoBERTa — {language}', f'xlmr_{language}')
save_evaluation_plots(test_df['label_id'].to_numpy(), all_predictions, all_probabilities,
                      'XLM-RoBERTa — all test languages', 'xlmr_all')

please 3.2 Attention visualization

This visualization displays the final XLM-RoBERTa layer's attention averaged across heads for one example. Attention weights are a diagnostic view of model attention, not proof of causal feature importance. The selected tweet is taken from the held-out test set and should not be used to tune the model.


In [ ]:
# Average last-layer attention across heads for one test example.
attention_example = test_df.sample(n=1, random_state=RANDOM_SEED).iloc[0]
attention_inputs = xlmr_tokenizer(
    attention_example['model_text'], return_tensors='pt',
    truncation=True, max_length=min(MAX_LENGTH, 64)
).to(xlmr_trainer.model.device)
xlmr_trainer.model.eval()
with torch.no_grad():
    attention_output = xlmr_trainer.model(**attention_inputs, output_attentions=True, return_dict=True)

if not attention_output.attentions:
    print('Attention tensors were not returned by this model/runtime; check eager attention support.')
else:
    last_layer_attention = attention_output.attentions[-1][0].mean(dim=0).detach().cpu().numpy()
    token_ids = attention_inputs['input_ids'][0].detach().cpu().tolist()
    token_labels = xlmr_tokenizer.convert_ids_to_tokens(token_ids)
    plt.figure(figsize=(max(8, len(token_labels) * 0.35), max(6, len(token_labels) * 0.30)), dpi=160)
    sns.heatmap(last_layer_attention, cmap='viridis', xticklabels=token_labels,
                yticklabels=token_labels)
    plt.title(f"Final-layer mean attention — {attention_example['language']}")
    plt.xlabel('Key token')
    plt.ylabel('Query token')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'xlmr_attention_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print('Attention example language:', attention_example['language'])

# Stage 4 — LSTM baseline

The baseline uses the same preprocessed text, label mapping, language-balanced training sample and validation sample as the transformer. It uses a train-only vocabulary, a bidirectional LSTM and gradient clipping. The held-out test sets are evaluated only after training.


In [ ]:
# Build a simple Unicode-aware whitespace/punctuation tokenizer and vocabulary from training text only.
TOKEN_PATTERN = re.compile(r'\w+|[^\w\s]', flags=re.UNICODE)
PAD_ID, UNK_ID = 0, 1
MAX_VOCAB_SIZE = 30000
LSTM_MAX_LENGTH = MAX_LENGTH


def simple_tokens(text):
    return TOKEN_PATTERN.findall(str(text).casefold())

train_token_lists = [simple_tokens(t) for t in balanced_train_df['model_text']]
word_counts = Counter(token for tokens in train_token_lists for token in tokens)
vocab = {'<PAD>': PAD_ID, '<UNK>': UNK_ID}
for token, _ in word_counts.most_common(MAX_VOCAB_SIZE - len(vocab)):
    if token not in vocab:
        vocab[token] = len(vocab)

print('LSTM vocabulary size:', len(vocab))


def encode_lstm_text(text, max_length=LSTM_MAX_LENGTH):
    ids = [vocab.get(tok, UNK_ID) for tok in simple_tokens(text)[:max_length]]
    return ids if ids else [UNK_ID]


class TweetTorchDataset(TorchDataset):
    def __init__(self, frame):
        self.texts = frame['model_text'].tolist()
        self.labels = frame['label_id'].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return torch.tensor(encode_lstm_text(self.texts[idx]), dtype=torch.long), self.labels[idx]


def lstm_collate(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_ID)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)


class BiLSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=96, dropout=0.3, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_ID)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=1)
        return self.classifier(self.dropout(final_hidden))


train_loader = DataLoader(TweetTorchDataset(balanced_train_df), batch_size=64,
                          shuffle=True, collate_fn=lstm_collate)
validation_loader = DataLoader(TweetTorchDataset(balanced_validation_df), batch_size=128,
                               shuffle=False, collate_fn=lstm_collate)
lstm_model = BiLSTMSentiment(len(vocab)).to(DEVICE)
train_label_counts = np.bincount(balanced_train_df['label_id'], minlength=NUM_LABELS)
lstm_class_weights = torch.tensor(
    len(balanced_train_df) / (NUM_LABELS * np.maximum(train_label_counts, 1)),
    dtype=torch.float32, device=DEVICE
)
lstm_optimizer = torch.optim.AdamW(lstm_model.parameters(), lr=1e-3, weight_decay=1e-4)
lstm_loss_fn = nn.CrossEntropyLoss(weight=lstm_class_weights)


def run_lstm_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    gold, probs_all = [], []
    for input_ids, lengths, labels in loader:
        input_ids, labels = input_ids.to(DEVICE), labels.to(DEVICE)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(input_ids, lengths)
            loss = lstm_loss_fn(logits, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        total_loss += float(loss.item()) * len(labels)
        gold.extend(labels.detach().cpu().numpy().tolist())
        probs_all.extend(torch.softmax(logits, dim=-1).detach().cpu().numpy().tolist())
    metrics, _ = metric_bundle(np.asarray(gold), np.asarray(probs_all))
    return total_loss / max(len(gold), 1), metrics, np.asarray(gold), np.asarray(probs_all)


best_val_f1 = -1.0
best_lstm_state = None
lstm_history = []
patience, stale_epochs = 2, 0
for epoch in range(1, 6):
    train_loss, train_metrics, _, _ = run_lstm_epoch(lstm_model, train_loader, lstm_optimizer)
    val_loss, val_metrics, _, _ = run_lstm_epoch(lstm_model, validation_loader)
    lstm_history.append({'epoch': epoch, 'train_loss': train_loss, 'validation_loss': val_loss,
                         'validation_accuracy': val_metrics['accuracy'],
                         'validation_f1_macro': val_metrics['f1_macro']})
    print(f"Epoch {epoch}/5 | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1={val_metrics['f1_macro']:.4f}")
    if val_metrics['f1_macro'] > best_val_f1:
        best_val_f1 = val_metrics['f1_macro']
        best_lstm_state = {k: v.detach().cpu().clone() for k, v in lstm_model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
    if epoch >= 3 and stale_epochs >= patience:
        print('Early stopping triggered.')
        break

if best_lstm_state is not None:
    lstm_model.load_state_dict(best_lstm_state)
lstm_model.to(DEVICE)
pd.DataFrame(lstm_history).to_csv(RESULTS_DIR / 'lstm_training_history.csv', index=False)


In [ ]:
# LSTM prediction and final test metrics

def predict_lstm(model, frame):
    loader = DataLoader(TweetTorchDataset(frame), batch_size=128, shuffle=False, collate_fn=lstm_collate)
    model.eval()
    gold, probabilities = [], []
    with torch.no_grad():
        for input_ids, lengths, labels in loader:
            logits = model(input_ids.to(DEVICE), lengths)
            gold.extend(labels.numpy().tolist())
            probabilities.extend(torch.softmax(logits, dim=-1).cpu().numpy().tolist())
    probabilities = np.asarray(probabilities)
    metrics, predictions = metric_bundle(np.asarray(gold), probabilities)
    return metrics, predictions, probabilities


lstm_metrics_rows = []
lstm_outputs = {}
for language in ['swahili', 'amharic', 'english']:
    subset = test_df[test_df['language'] == language].reset_index(drop=True)
    metrics, predictions, probabilities = predict_lstm(lstm_model, subset)
    lstm_metrics_rows.append({'model':'BiLSTM', 'evaluation':f'{language}_test', 'n':len(subset), **metrics})
    lstm_outputs[language] = (subset, predictions, probabilities)

lstm_all_metrics, lstm_all_predictions, lstm_all_probabilities = predict_lstm(
    lstm_model, test_df.reset_index(drop=True)
)
lstm_metrics_rows.append({'model':'BiLSTM', 'evaluation':'all_test', 'n':len(test_df), **lstm_all_metrics})
lstm_metrics_df = pd.DataFrame(lstm_metrics_rows)
display(lstm_metrics_df)
lstm_metrics_df.to_csv(RESULTS_DIR / 'lstm_test_metrics.csv', index=False)

for language, (subset, predictions, probabilities) in lstm_outputs.items():
    save_evaluation_plots(subset['label_id'].to_numpy(), predictions, probabilities,
                          f'BiLSTM — {language}', f'lstm_{language}')
save_evaluation_plots(test_df['label_id'].to_numpy(), lstm_all_predictions,
                      lstm_all_probabilities, 'BiLSTM — all test languages', 'lstm_all')

comparison_df = pd.concat([xlmr_metrics_df, lstm_metrics_df], ignore_index=True)
display(comparison_df)
comparison_df.to_csv(RESULTS_DIR / 'model_comparison_metrics.csv', index=False)


# Stage 5 — Ablation study

The base configuration is compared with one-factor-at-a-time changes to batch size, learning rate and maximum sequence length. These experiments use the balanced training and validation samples only; the test data are not used to select settings. Each run uses the same early-stopping and gradient-clipping controls as the main model. This section is computationally expensive, so it has an explicit switch.


In [ ]:
RUN_ABLATIONS = True
ABLATION_EPOCHS = 5
ABLATION_CONFIGS = [
    {'run_name':'ablation_batch16', 'batch_size':16, 'learning_rate':2e-5, 'max_length':128,
     'changed_factor':'batch_size', 'setting':'16'},
    {'run_name':'ablation_lr3e5', 'batch_size':8, 'learning_rate':3e-5, 'max_length':128,
     'changed_factor':'learning_rate', 'setting':'3e-5'},
    {'run_name':'ablation_seq96', 'batch_size':8, 'learning_rate':2e-5, 'max_length':96,
     'changed_factor':'max_length', 'setting':'96'},
]

ablation_rows = [{'run_name':'xlmr_main', 'changed_factor':'baseline', 'setting':'batch=8; lr=2e-5; max_length=128',
                  'batch_size':MAIN_BATCH_SIZE, 'learning_rate':MAIN_LEARNING_RATE,
                  'max_length':MAX_LENGTH, 'validation_f1_macro':float(xlmr_trainer.evaluate().get('eval_f1_macro', np.nan))}]

if RUN_ABLATIONS:
    for config in ABLATION_CONFIGS:
        print('\nRunning ablation:', config['run_name'])
        ablation_trainer, ablation_tokenizer = train_xlmr(
            balanced_train_df, balanced_validation_df,
            run_name=config['run_name'], batch_size=config['batch_size'],
            learning_rate=config['learning_rate'], max_length=config['max_length'],
            epochs=ABLATION_EPOCHS,
        )
        eval_metrics = ablation_trainer.evaluate()
        ablation_rows.append({
            'run_name':config['run_name'], 'changed_factor':config['changed_factor'],
            'setting':config['setting'], 'batch_size':config['batch_size'],
            'learning_rate':config['learning_rate'], 'max_length':config['max_length'],
            'validation_f1_macro':float(eval_metrics.get('eval_f1_macro', np.nan)),
        })
        del ablation_trainer, ablation_tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)
ablation_df.to_csv(RESULTS_DIR / 'ablation_results.csv', index=False)
plt.figure(figsize=(9, 5), dpi=160)
sns.barplot(data=ablation_df, x='run_name', y='validation_f1_macro')
plt.title('Ablation study: validation macro-F1')
plt.xlabel('Experiment')
plt.ylabel('Validation macro-F1')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'ablation_validation_f1.png', dpi=300, bbox_inches='tight')
plt.show()

# Stage 6 — Cross-lingual transfer

Train a fresh XLM-RoBERTa model using **Swahili training data only**, select it using Swahili validation data, then evaluate on the untouched Swahili, English and Amharic test subsets. English and Amharic test examples are not used for training or validation in this experiment. Because the English data source is TweetEval, cross-dataset/domain differences are part of the interpretation.


In [ ]:
RUN_CROSSLINGUAL = True
if RUN_CROSSLINGUAL:
    sw_train = model_data[(model_data['language'] == 'swahili') & (model_data['source_split'] == 'train')].copy()
    sw_validation = model_data[(model_data['language'] == 'swahili') & (model_data['source_split'] == 'validation')].copy()
    cross_trainer, cross_tokenizer = train_xlmr(
        sw_train, sw_validation, run_name='xlmr_swahili_only',
        batch_size=8, learning_rate=2e-5, max_length=MAX_LENGTH, epochs=5,
    )

    cross_rows = []
    cross_outputs = {}
    for language in ['swahili', 'english', 'amharic']:
        subset = test_df[test_df['language'] == language].reset_index(drop=True)
        metrics, predictions, probabilities = predict_xlmr(cross_trainer, cross_tokenizer, subset)
        cross_rows.append({'training_language':'swahili', 'evaluation_language':language,
                           'n':len(subset), **metrics})
        cross_outputs[language] = (subset, predictions, probabilities)
        save_evaluation_plots(subset['label_id'].to_numpy(), predictions, probabilities,
                              f'Swahili-trained XLM-R — {language} test', f'crosslingual_sw_train_{language}')

    crosslingual_df = pd.DataFrame(cross_rows)
    display(crosslingual_df)
    crosslingual_df.to_csv(RESULTS_DIR / 'crosslingual_sw_train_metrics.csv', index=False)


# Stage 7 — Reproducibility summary and artifact checks

This final cell saves the measured metrics, actual training configuration and figure inventory. It does not substitute missing results with estimates. If any section was skipped (for example `RUN_ABLATIONS=False`), the corresponding output file will be absent and should be reported as not run.


In [ ]:
# Persist the experiment metadata and list all generated figure files.
experiment_summary = {
    'project': 'Group 3 Multilingual Tweet Sentiment Analysis',
    'dataset_provenance': {'amharic':'AfriSenti', 'swahili':'AfriSenti', 'english':'TweetEval sentiment'},
    'canonical_labels': LABEL2ID,
    'main_model': XLMR_NAME,
    'baseline_model': 'Bidirectional LSTM',
    'main_training': {
        'max_epochs': MAIN_EPOCHS,
        'early_stopping_patience': 2,
        'gradient_clip_norm': 1.0,
        'batch_size': MAIN_BATCH_SIZE,
        'learning_rate': MAIN_LEARNING_RATE,
        'max_sequence_length': MAX_LENGTH,
        'balanced_languages': True,
    },
    'normalization_replacement_totals': dict(normalization_totals),
    'run_ablations': RUN_ABLATIONS,
    'run_crosslingual': RUN_CROSSLINGUAL,
    'device': str(DEVICE),
    'generated_utc': datetime.now(timezone.utc).isoformat(),
    'figures': sorted(p.name for p in FIGURES_DIR.glob('*.png')),
}
with open(RESULTS_DIR / 'experiment_summary.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_summary, f, ensure_ascii=False, indent=2)

print('Experiment results folder:', RESULTS_DIR)
print('Generated figure files:')
for p in sorted(FIGURES_DIR.glob('*.png')):
    print(f' - {p.name} ({p.stat().st_size:,} bytes)')
print('Result tables:')
for p in sorted(RESULTS_DIR.glob('*.csv')):
    print(f' - {p.name} ({p.stat().st_size:,} bytes)')